# PyPSA: Introduction

<div style="text-align: center;">
  <iframe src="https://mozilla.github.io/pdf.js/web/viewer.html?file=https://raw.githubusercontent.com/fneum/nordnet/main/_static/lecture-nordnet-markets.pdf" width="100%" height="600" frameborder="0"></iframe>
</div>

PyPSA is an open source Python package for simulating and optimising modern energy systems that include features such as

- conventional generators with unit commitment (ramp-up, ramp-down, start-up, shut-down),
- time-varying wind and solar generation,
- energy storage with efficiency losses and inflow/spillage for hydroelectricity
- coupling to other energy sectors (electricity, transport, heat, industry),
- conversion between energy carriers (e.g. electricity to hydrogen),
- transmission networks (AC, DC, other fuels)

PyPSA can be used for a variety of problem types (e.g. electricity market modelling, security of supply analysis, long-term investment planning under uncertainty, transmission network expansion planning), and is designed to scale well with large networks and long time series.

**PyPSA does the following things for you:**

- manage data inputs in standardised format
- build standardised optimisation problem
- communicate with the solver(s)
- retrieve and process optimisation results
- manage data outputs in standardised format

:::{note}
Documentation for this package is available at https://docs.pypsa.org.
:::

:::{note}
If you have not yet set up Python on your computer, you can execute this tutorial in your browser via [Google Colab](https://colab.research.google.com/). Download the `.ipynb` file using the download button on the top right corner and import it in [Google Colab](https://colab.research.google.com/).

Then install the following packages by executing the following command in a Jupyter cell at the top of the notebook.

```sh
!pip install pypsa"<1.2" matplotlib cartopy
```
:::

## Basic Structure

| Component | Description |
| --- | --- |
| [Network](https://docs.pypsa.org/latest/user-guide/design/#network-components) | Container for all components. |
| [Bus](https://docs.pypsa.org/latest/user-guide/components/buses/) | Node where components attach. |
| [Carrier](https://docs.pypsa.org/latest/user-guide/components/carriers/) | Energy carrier or technology (e.g. electricity, hydrogen, gas, coal, oil, biomass, on-/offshore wind, solar). Can track properties such as specific carbon dioxide emissions or nice names and colors for plots. |
| [Load](https://docs.pypsa.org/latest/user-guide/components/loads/) | Energy consumer (e.g. electricity demand). |
| [Generator](https://docs.pypsa.org/latest/user-guide/components/generators/) | Generator (e.g. power plant, wind turbine, PV panel). |
| [Line](https://docs.pypsa.org/latest/user-guide/components/lines/) | Power distribution and transmission lines (overhead and cables). |
| [Link](https://docs.pypsa.org/latest/user-guide/components/links/) | Links connect two buses with controllable energy flow, direction-control and losses. They can be used to model: <ul><li>HVDC links</li><li>HVAC li'nes (neglecting KVL, only net transfer capacities (NTCs))</li><li>conversion between carriers (e.g. electricity to hydrogen in electrolysis)</li></ul> |
| [StorageUnit](https://docs.pypsa.org/latest/user-guide/components/storage-units/) | Storage with fixed nominal energy-to-power ratio. |
| [GlobalConstraint](https://docs.pypsa.org/latest/user-guide/components/global-constraints/) | Constraints affecting many components at once, such as emission limits. |
| [Store](https://docs.pypsa.org/latest/user-guide/components/stores/) | Storage with separately extendable energy capacity. |


:::{note}
Links in the table lead to documentation for each component.
:::

<img src="https://docs.pypsa.org/latest/assets/images/buses.png" width="500px" />


:::{warning}
Per unit values of voltage and impedance are used internally for network calculations. It is assumed internally that the base power is **1 MW**.
:::

## Model Formulation

**The design principle of PyPSA is that each component is associated with a set of variables and constraints that will be added to the optimisation model based on the input data stored for the components.**

:::{note}
For a full reference to the optimisation problem description, see https://docs.pypsa.org/latest/user-guide/optimization/overview/
:::

## Electricity market example

Let's get acquainted with PyPSA to build a simple electricity market model.

We have the following data:

- fuel costs in € / MWh$_{th}$ 

In [ ]:
fuel_cost = dict(
    coal=8,
    gas=100,
    oil=48,
)

- efficiencies of thermal power plants in MWh$_{el}$ / MWh$_{th}$

In [ ]:
efficiency = dict(
    coal=0.33,
    gas=0.58,
    oil=0.35,
)

- specific emissions in t$_{CO_2}$ / MWh$_{th}$

In [ ]:
# t/MWh thermal
emissions = dict(
    coal=0.34,
    gas=0.2,
    oil=0.26,
    hydro=0,
    wind=0,
)

- power plant capacities in MW

In [ ]:
power_plants = {
    "A": {"coal": 35000, "wind": 3000, "gas": 8000, "oil": 2000},
    "B": {"hydro": 1200},
}

- electrical load in MW

In [ ]:
loads = {
    "A": 42000,
    "B": 650,
}

## Building a network

By convention, PyPSA is imported without an alias:

In [ ]:
import pypsa

In [ ]:
pypsa.options.params.optimize.include_objective_constant = True

First, we create a new network object which serves as the overall container for all components.

In [ ]:
n = pypsa.Network()

The second component we need are buses. **Buses** are the fundamental nodes of the network, to which all other components like loads, generators and transmission lines attach. They enforce energy conservation for all elements feeding in and out of it (i.e. Kirchhoff’s Current Law).

Components can be added to the network `n` using the `n.add()` function. It takes the component name as a first argument, the name of the component as a second argument and possibly further parameters as keyword arguments. Let's use this function, to add buses for each country to our network:

In [ ]:
n.add("Bus", "A", v_nom=400)
n.add("Bus", "B", v_nom=400)

For each class of components, the data describing the components is stored in a `pandas.DataFrame`. For example, all static data for buses is stored in `n.buses`

In [ ]:
n.buses

You see there are many more attributes than we specified while adding the buses; many of them are filled with default parameters which were added. You can look up the field description, defaults and status (required input, optional input, output) for buses here https://docs.pypsa.org/latest/user-guide/components/buses/, and analogously for all other components. 

The method `n.add()` also allows you to add multiple components at once. For instance, multiple **carriers** for the fuels with information on specific carbon dioxide emissions, a nice name, and colors for plotting. For this, the function takes the component name as the first argument and then a list of component names and then optional arguments for the parameters. Here, scalar values, lists, dictionary or `pandas.Series` are allowed. The latter two needs keys or indices with the component names.

In [ ]:
n.add(
    "Carrier",
    ["coal", "gas", "oil", "hydro", "wind"],
    co2_emissions=emissions,
    nice_name=["Coal", "Gas", "Oil", "Hydro", "Onshore Wind"],
    color=["dimgrey", "tomato", "olive", "seagreen", "royalblue"],
)

n.add("Carrier", "AC", nice_name="Electricity", color="crimson")

The `n.add()` function is very general. It lets you add any component to the network object `n`. For instance, in the next step we add **generators** for all the different power plants.

In Region B:

In [ ]:
n.add(
    "Generator",
    "B hydro",
    bus="B",
    carrier="hydro",
    p_nom=1200,  # MW
    marginal_cost=0,  # default
)

In Region A (in a loop):

In [ ]:
for tech, p_nom in power_plants["A"].items():
    n.add(
        "Generator",
        f"A {tech}",
        bus="A",
        carrier=tech,
        efficiency=efficiency.get(tech, 1),
        p_nom=p_nom,
        marginal_cost=fuel_cost.get(tech, 0) / efficiency.get(tech, 1),
    )

As a result, the `n.generators` DataFrame looks like this:

In [ ]:
n.generators

Next, we're going to add the electricity demand. This time, without a loop.

A positive value for `p_set` means consumption of power from the bus (in MW).

In [ ]:
n.add(
    "Load",
    "A electricity demand",
    bus="A",
    p_set=loads["A"],
    carrier="AC",
)

In [ ]:
n.add(
    "Load",
    "B electricity demand",
    bus="B",
    p_set=loads["B"],
    carrier="AC",
)

In [ ]:
n.loads

Finally, we add the connection between Region B and Region A with a 500 MW line:

In [ ]:
n.add(
    "Line",
    "A-B",
    bus0="A",
    bus1="B",
    s_nom=500,
    x=1,
    r=1,
)

In [ ]:
n.lines

## Optimisation

With all input data transferred into PyPSA's data structure, we can now build and run the resulting optimisation problem. In PyPSA, building, solving and retrieving results from the optimisation model is contained in a single function call `n.optimize()`. This function optimizes the operational dispatch decisions for least cost, as well as investment decisions for components marked as extendable *(more on this in the next section)*.

The `n.optimize()` function can take many arguments. The most relevant for the moment is the choice of the solver (e.g. "highs" and "gurobi"). They need to be installed to use them here:

In [ ]:
n.optimize(solver_name="highs", log_to_console=False)

Let's have a look at the results.

Since the power flow and dispatch are generally time-varying quantities, these are stored in a different location than e.g. `n.generators`. They are stored in `n.generators_t`. Thus, to find out the dispatch of the generators, run

In [ ]:
n.generators_t.p

or if you prefer it in relation to the generators nominal capacity

In [ ]:
n.generators_t.p / n.generators.p_nom

You see that the time index has the value 'now'. This is the default index when no time series data has been specified and the network only covers a single state (e.g. a particular hour). 

Similarly you will find the power flow in transmission lines at

In [ ]:
n.lines_t.p0

In [ ]:
n.lines_t.p1

The `p0` will tell you the flow from `bus0` to `bus1`. `p1` will tell you the flow from `bus1` to `bus0`.

What about the shadow prices?

In [ ]:
n.buses_t.marginal_price

## Model inspection

Under the hood, a call to `n.optimize()` builds an optimisation model, solves it with the specified solver, and then retrieves the results back into the PyPSA data structure. We can access this intermediate model via `n.model`. This allows us to explore the optimisation model in more detail, for instance to see all variables and constraints that were added to the optimisation problem.

In [ ]:
n.model

In [ ]:
n.model.constraints["Generator-fix-p-upper"]

In [ ]:
n.model.constraints["Bus-nodal_balance"]

In [ ]:
n.model.objective

In [ ]:
n.model.constraints["Generator-fix-p-upper"].dual.to_dataframe()

## Modifying networks

Modifying data of components in an existing PyPSA network is as easy as modifying the entries of a `pandas.DataFrame`. For instance, if we want to reduce the cross-border transmission capacity between Region A and Region B, we'd run:

In [ ]:
n.lines.loc["A-B", "s_nom"] = 400
n.lines

In [ ]:
n.optimize(log_to_console=False)

You can see that the production of the hydro power plant was reduced and that of the gas power plant increased owing to the reduced transmission capacity.

In [ ]:
n.generators_t.p

## Global constraints for emission limits

In the example above, we happen to have some spare gas capacity with lower carbon intensity than the coal and oil generators. We could use this to lower the emissions of the system, but it will be more expensive. We can implement the limit of carbon dioxide emissions as a constraint.

This is achieved in PyPSA through **Global Constraints** which add constraints that apply to many components at once.

But first, we need to calculate the current level of emissions to set a sensible limit.

We can compute the emissions per generator (in tonnes of CO$_2$) in the following way.

$$\frac{g_{i,s,t} \cdot \rho_{i,s}}{\eta_{i,s}}$$

where $ \rho$ is the specific emissions (tonnes/MWh thermal) and $\eta$ is the conversion efficiency (MWh electric / MWh thermal) of the generator with dispatch $g$ (MWh electric):

In [ ]:
e = (
    n.generators_t.p
    / n.generators.efficiency
    * n.generators.carrier.map(n.carriers.co2_emissions)
)
e

Summed up, we get total emissions in tonnes:

In [ ]:
e.sum().sum()

So, let's say we want to reduce emissions by 10%:

In [ ]:
n.add(
    "GlobalConstraint",
    "emission_limit",
    carrier_attribute="co2_emissions",
    sense="<=",
    constant=e.sum().sum() * 0.9,
)

In [ ]:
n.optimize(log_to_console=False)

In [ ]:
n.generators_t.p

In [ ]:
n.generators_t.p / n.generators.p_nom

In [ ]:
n.global_constraints.mu

Can we lower emissions even further? Say by another 5% points?

In [ ]:
n.global_constraints.loc["emission_limit", "constant"] = 0.85

In [ ]:
n.optimize(log_to_console=False)

**No!** Without any additional capacities, we have exhausted our options to reduce emissions in that hour. The solver tells us that the problem is *infeasible*, i.e. there is no solution that satisfies all constraints. Better revert this change:

In [ ]:
n.global_constraints.loc["emission_limit", "constant"] = 0.9

## Example Grid Model with Time Resolution

**Dispatch problem with German SciGRID network**


[SciGRID](https://www.dlr.de/en/ve/research-and-transfer/projects/project-scigrid) is a project that provides an open reference model of the European transmission network. The network comprises time series for loads and the availability of renewable generation at an hourly resolution for January 1, 2011 as well as approximate generation capacities in 2014. This dataset is a little out of date and only intended to demonstrate the capabilities of PyPSA.

In [ ]:
n2 = pypsa.examples.scigrid_de()

There are some infeasibilities without allowing extension. Also, to approximate so-called $N-1$ security, we don't allow any line to be loaded above 70% of their thermal rating. $N-1$ security is a constraint that states that no single transmission line may be overloaded by the failure of another transmission line (e.g. through a tripped connection).

In [ ]:
n2.lines.s_max_pu = 0.7
n2.lines.loc[["316", "527", "602"], "s_nom"] = 1715

Because this network includes time-varying data, now is the time to look at another attribute of `n`: `n.snapshots`. Snapshots is the PyPSA terminology for time steps. In most cases, they represent a particular hour. They can be a `pandas.DatetimeIndex` or any other list-like attributes.

In [ ]:
n2.snapshots[:4]

This index will match with any time-varying attributes of components:

In [ ]:
n2.loads_t.p_set.iloc[:3, :3]

We can use simple `pandas` syntax, to create an overview of the load time series...

In [ ]:
n2.loads_t.p_set.sum(axis=1).plot(ylim=[0, 60e3], ylabel="MW")

... and the capacity factor time series:

In [ ]:
n2.generators_t.p_max_pu.T.groupby(n2.generators.carrier).mean().T.plot(ylabel="p.u.")

We can also inspect the total power plant capacities per technology.

In [ ]:
n2.generators.groupby("carrier").p_nom.sum().div(1e3).sort_values().plot.barh(
    xlabel="GW"
)

The network inputs and outputs can also be visualised on a map. The `n.plot()` function uses `matplotlib` to create static plots, while `n.explore()` creates interactive plots based on `pydeck`. In the following, we will focus on the interactive plotting with `n.explore()`.

The `n.explore()` function has a variety of styling arguments to tweak the appearance of the buses, the lines and the map in the background. For example, we can size the buses according to their load:

In [ ]:
load = n2.loads_t.p_set.sum(axis=0).groupby(n2.loads.bus).sum()
load.head(3)

In [ ]:
n2.explore(bus_size=load / 20)

The `bus_size` argument of `n.explore()` can be even more powerful. It can show the nodal composition of power plant capacities as pie charts by grouping data by the `bus` and `carrier` attributes of the generators:

In [ ]:
capacities = n2.generators.groupby(["bus", "carrier"]).p_nom.sum()
capacities.head(3)

... for which we need to assign some colors to the technologies first:

In [ ]:
colors = {
    "Gas": "tomato",
    "Hard Coal": "dimgrey",
    "Run of River": "turquoise",
    "Waste": "olive",
    "Brown Coal": "peru",
    "Oil": "black",
    "Storage Hydro": "teal",
    "Other": "whitesmoke",
    "Multiple": "whitesmoke",
    "Nuclear": "deeppink",
    "Geothermal": "darkorange",
    "Wind Offshore": "lightskyblue",
    "Wind Onshore": "royalblue",
    "Solar": "gold",
    "Pumped Hydro": "lightseagreen",
    "AC": "crimson",
}
n2.add("Carrier", colors.keys(), color=colors.values(), overwrite=True)

Note that if you do not mind about the individual colors of a carrier, you can also let `n.sanitize()` do the magic and auto-assign colors for you. This can be helpful if you just want to have a quickly explorable, fully functioning network without spending much time on the cosmetic side. Next to mapping colors, `n.sanitize()` also "heals" the network by adding missing buses or carriers, if there are any.

In [ ]:
n2.explore(bus_size=capacities / 3)

So let's solve the electricity market simulation for January 1, 2011:

In [ ]:
n2.optimize(log_to_console=False)

Now, we can also plot model outputs, like the calculated power flows on the network map using the `line_flow` argument of `n.explore()`:

In [ ]:
line_loading = (
    n2.lines_t.p0.iloc[0].abs() / n2.lines.s_nom / n2.lines.s_max_pu * 100
)  # %

In [ ]:
n2.explore(
    bus_size=1e1,
    line_color=line_loading.abs(),
    line_flow=n2.lines_t.p0.iloc[0] / 100,
    line_cmap="plasma",
    line_width=n2.lines.s_nom / 1000,
)

Or plot the hourly dispatch using PyPSA's built-in statistics functionality:

In [ ]:
n2.statistics.energy_balance.iplot()

## Statistics

The PyPSA `n.statistics` module provides a variety of pre-defined tables and plots to analyse optimisation results at different aggregation levels. For instance, the energy balance, the operational costs (OPEX), curtailment, prices and market value. We will use this module in more detail in later workshops on sector coupling and investment optimisation.

Energy balance:

In [ ]:
n2.statistics.energy_balance().div(1e3).round(1).sort_values()

Operational costs (OPEX):

In [ ]:
n2.statistics.opex().round(1).sort_values(ascending=False)

There is much more to explore in PyPSA. If you are hooked, have a look at the [documentation](https://docs.pypsa.org/) and the [examples section](https://docs.pypsa.org/latest/examples/examples/).

Currently supported metrics in `n.statistics` are:

- [Capital expenditure](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.capex): The capital expenditure of all components.
- [Installed capital expenditure](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.installed_capex): The capital expenditure of all components before optimization.
- [Expanded capital expenditure](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.expanded_capex): The capital expenditure of all components added during optimization.
- [Overnight investment cost](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.overnight_cost): The overnight investment costs (excluding FOM) of all components.
- [Fixed O&M](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.fom): The fixed operation and maintenance costs of all components.
- [Operational expenditure](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.opex): The operational expenditure of all components.
- [Installed capacities](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.installed_capacity): The capacities of all components before optimization.
- [Expanded capacities](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.expanded_capacity): The capacities of all components added during optimization.
- [Optimal capacities](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.optimal_capacity): The total capacities of all components after optimization.
- [Supply](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.supply): The energy supplied by all components.
- [Withdrawal](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.withdrawal): The energy withdrawn by all components.
- [Curtailment](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.curtailment): The energy curtailed in all components.
- [Capacity Factor](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.capacity_factor): The capacity factor / utilization rate of all components.
- [Revenue](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.revenue): The revenue received by all components.
- [Market value](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.market_value): The market value of all components.
- [Energy balance](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.energy_balance): The energy balance of the network across all carriers and snapshots.
- [System costs](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.system_cost): The total system costs after optimization, including capital and operational expenditure.
- [Marginal prices](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.prices): The marginal prices at the buses for all snapshots.
- [Transmission](https://docs.pypsa.org/latest/api/networks/statistics/#pypsa.Network.statistics.transmission): The energy transmitted through transmission components (links, lines, transformers connecting to buses of the same carrier).